### Part 1: What is an LLM Gateway?
Think of an LLM Gateway as a smart middleware layer that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

                    ┌─────────────────────────────┐
                    │       Your Application      │
                    │  (Chatbot, RAG, Agent, etc) │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │       LLM GATEWAY           │
                    │  • Routing                  │
                    │  • Fallbacks                │
                    │  • Caching                  │
                    │  • Rate Limiting            │
                    │  • Cost Tracking            │
                    │  • Observability            │
                    └──────┬─────┬─────┬─────┬────┘
                           │     │     │     │
                           ▼     ▼     ▼     ▼
                        OpenAI Claude Gemini Groq
### Without a Gateway (The Pain)
- Different SDKs and APIs for every provider
- No fallback if one provider goes down
- No central place to track costs
- Hard to switch models without rewriting code
- No caching → paying twice for the same query
### With a Gateway (The Joy)
- One unified API for 100+ providers
- Automatic fallbacks if a provider fails
- Centralized logging, cost tracking, rate limiting
- Swap models with a config change, no code rewrite
- Cache repeated queries → save money

In [1]:
import warnings 
import logging 

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import Litellm normally
from litellm import completion 


In [2]:
import litellm
litellm.suppress_debug_info = True

In [3]:
import warnings
import logging

# Keep the recording clean — suppress noisy AWS-related warnings
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [ ]:
# Load API keys from a .env file
# Create a .env file in the same folder with:

import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
# print("OpenAI key loaded:    ",os.getenv("OPENAI_API_KEY"))
# print("Anthropic key loaded: ",os.getenv("ANTHROPIC_API_KEY"))
# print("Groq key loaded:      ",os.getenv("GROQ_API_KEY"))

True

### The Simplest LiteLLM Example — Unified API
The biggest pain point: every provider has a different SDK.

LiteLLM gives you one function — completion() — that works with all of them. Look at how clean this is:

In [5]:
from litellm import completion

# Same code, different providers — just change the `model` string!

# Call OpenAI
from litellm import completion
import os

response = completion(
    model="gemini/gemini-2.5-flash",
    messages=[
        {"role": "user", "content": "Explain RAG in one sentence."}
    ]
)

print(response.choices[0].message.content)



# Call Groq (super fast inference)
response_groq = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("Groq:      ", response_groq.choices[0].message.content)

RAG enhances large language models by retrieving relevant external data to ground their generated responses, improving accuracy and reducing hallucinations.
Groq:       RAG (Retrieve, Augment, Generate) is a type of natural language processing (NLP) model that uses a combination of retrieval and generation techniques to answer questions and provide information by fetching relevant context, augmenting it with additional knowledge, and then generating a response.


In [6]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("🔵 OpenAI",     "gpt-4o-mini"),
    ("🟢 Groq",       "groq/llama-3.3-70b-versatile"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-1.5-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")

🔵 OpenAI       : ❌ RateLimitError
🟢 Groq         : RAG (Retrieval, Augment, Generate) is a type of artificial intelligence model th
🟣 Anthropic    : ❌ BadRequestError
🟡 Gemini       : ❌ NotFoundError


### Automatic Fallbacks — When OpenAI Goes Down
Real story: OpenAI had a 4-hour outage in November 2023. Apps that hard-coded gpt-4 went completely dark.

With a gateway, if one provider fails, we automatically fall back to another. Production apps must have this.

In [7]:
from litellm import completion

# Define a fallback chain: try GPT first, then Claude, then Groq
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=["groq/llama-3.3-70b-versatile",
        "gpt-4o-mini"
        
    ]
)

print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

21:47:31 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.NotFoundError: GeminiException - {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}
Traceback (most recent call last):
  File "c:\Users\bharat singh thakur\Desktop\Python\LangChainNew\.venv\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 3100, in async_completion
    response = await client.post(
               ^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )  # type: ignore
    ^
  File "c:\Users\bharat singh thakur\Desktop\Python\LangChainNew\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 297, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^

Response: An LLM (Local Loop Maintenance) Gateway, also known as an LLM gateway or LLM proxy, is a device or software that acts as an intermediary between a Local Loop (the connection between a customer's premi ...

Which model actually answered? llama-3.3-70b-versatile


### Cost Tracking — Know Where Your Money Goes
LiteLLM automatically calculates the cost of every call using its built-in pricing database. No more surprise bills.

In [ ]:
from litellm import completion, completion_cost

response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": "Write a haiku about AI."}]
)

# Get the exact USD cost of this single call
cost = completion_cost(completion_response=response)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



RateLimitError: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.